# Chapter 8 – Cost Optimization
## Hands-On Colab Notebook for Network Engineers

**The Friday afternoon Slack message:**

> *"Ed, I need to explain the 'AI Services' line item on this month's cloud bill.*
> *It says $8,500. When we approved this project the budget was $500/month."*
>
> *After I exported the usage logs, the breakdown was embarrassing:*
> *43% — developers testing the same configs again and again*
> *28% — production using Opus for simple log classification (WHY?)*
> *12% — a retry loop bug hammering the API on failures*
> *11% — legitimate production workload (the actual reason this exists)*
> *Only 11% of the $8,500 was doing what we built this system to do.*

| # | Demo | What it teaches |
|---|------|-----------------|
| 1 | Token Economics | The cost formula — input vs output pricing |
| 2 | Cost Tracker | Wrap every call — see exactly who spent what |
| 3 | Prompt Optimization | Verbose vs tight — same quality, 75% fewer tokens |
| 4 | Model Routing (Cascade) | Route simple → Haiku, complex → Sonnet. Save 60-80% |
| 5 | Batch API | 50% discount for non-urgent jobs (nightly audits) |
| 6 | Semantic Caching | Same question twice = zero API calls the second time |
| 7 | Budget Guard | Hard daily limits per user — no more surprise bills |

**~35 minutes** | **~$0.08 in API calls** | **No local setup required**

> Networking analogy: AI cost optimization = QoS.
> Classification → model routing (DSCP marking → queue assignment).
> Budget enforcement → spending policing (CoPP for your AI budget).
> Semantic caching → ARP cache (don't broadcast the same question twice).


---
## Setup

In [ ]:
!pip install anthropic -q

import json, math, time
from anthropic import Anthropic
from dataclasses import dataclass, field
from datetime import datetime

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic(api_key=ANTHROPIC_API_KEY)

HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"
OPUS   = "claude-opus-4-20250115"

# Current pricing in $/million tokens
PRICING = {
    HAIKU:  {"in": 1.00,  "out": 5.00},
    SONNET: {"in": 3.00,  "out": 15.00},
    OPUS:   {"in": 15.00, "out": 75.00},
}

def calc_cost(model, in_tok, out_tok, cache_read=0, cache_write=0):
    """Calculate exact cost for a set of token counts."""
    p = PRICING.get(model, PRICING[SONNET])
    regular   = max(0, in_tok - cache_read - cache_write)
    return (regular            / 1_000_000 * p["in"]
          + cache_write        / 1_000_000 * p["in"] * 1.25
          + cache_read         / 1_000_000 * p["in"] * 0.10
          + out_tok            / 1_000_000 * p["out"])

print("Setup complete.")
print(f"{'Model':<10} {'Input $/M':>10} {'Output $/M':>11} {'Output/Input':>13}")
print("-" * 48)
for m, p in PRICING.items():
    ratio = p['out'] / p['in']
    print(f"  {m.split('-')[1]:<8} ${p['in']:>9.2f} ${p['out']:>10.2f} {ratio:>12.0f}x")
print()
print("Key insight: output tokens cost 5x more than input tokens.")
print("Every token you save on OUTPUT is worth 5x more than saving an input token.")


---
## Demo 1 — Token Economics: The Cost Formula

**Before optimizing anything, understand exactly what you are paying for.**

The formula for any API call:
```
Cost = (input_tokens  / 1,000,000 × input_price)
     + (output_tokens / 1,000,000 × output_price)
```

This cell reconstructs the $8,500 mystery bill from the opening story.
Same formula — different call volumes and models — explains everything.


In [ ]:
# ── Reconstruct the $8,500 bill ──────────────────────────────────────────────
# Each row: (label, model, in_tok, out_tok, calls_per_month)
bill_items = [
    ("Dev testing (same configs)",   HAIKU,  3_000,  800, 1_800),
    ("Prod: log classify on OPUS",   OPUS,     500,   50, 3_500),
    ("Retry loop bug",               SONNET, 2_000,  400, 2_000),
    ("Legitimate production work",   SONNET, 2_500,  500,   400),
    ("Chatbot (out-of-scope Qs)",    SONNET, 1_200,  600,   250),
]

total = 0.0
print(f"{'Source':<30} {'Model':<8} {'Calls':>6} {'$/call':>8} {'$/month':>10} {'%':>5}")
print("─" * 72)
for label, model, in_t, out_t, calls in bill_items:
    cost_per = calc_cost(model, in_t, out_t)
    monthly  = cost_per * calls
    total   += monthly
    model_s  = model.split("-")[1]
    print(f"{label:<30} {model_s:<8} {calls:>6,} ${cost_per:>7.4f} ${monthly:>9.2f}")

print("─" * 72)
print(f"{'TOTAL':<30} {'':8} {'':>6} {'':>8} ${total:>9.2f}")
print()

# ── What the optimized bill looks like after fixes ───────────────────────────
print("After optimization (2 weeks later):")
fixed_items = [
    ("Dev testing (result caching)", HAIKU,  3_000,  800,   200),   # 90% less testing
    ("Prod: log classify on HAIKU",  HAIKU,    500,   50, 3_500),   # Moved to Haiku
    ("Retry loop fixed",             SONNET, 2_000,  400,     0),   # Bug fixed
    ("Legitimate production work",   SONNET, 2_500,  500,   400),   # Unchanged
    ("Chatbot removed",              HAIKU,    500,  100,     0),   # Removed feature
]

fixed_total = 0.0
for label, model, in_t, out_t, calls in fixed_items:
    cost_per     = calc_cost(model, in_t, out_t)
    fixed_total += cost_per * calls

print(f"  Before: ${total:,.2f}/month")
print(f"  After:  ${fixed_total:,.2f}/month")
print(f"  Saving: ${total - fixed_total:,.2f}/month ({(total-fixed_total)/total*100:.0f}% reduction)")
print()
print("Same production workload. Less waste. Less wrong model choice. Less bugs.")


---
## Demo 2 — Cost Tracker: See Who Spent What

**You cannot optimize what you have not measured.**

The `CostTracker` wraps every API call and records:
- Which model was used
- How many tokens were consumed
- Exact cost in USD
- Which feature/team triggered the call

After a week of logging, the biggest cost centers become obvious.

> Networking analogy: NetFlow / IPFIX.
> You don't guess which traffic is consuming your WAN bandwidth.
> You collect flow data and look at the report.


In [ ]:
@dataclass
class CallRecord:
    timestamp:   str
    model:       str
    task_type:   str    # what was this call for? (security_audit, log_classify, etc.)
    in_tokens:   int
    out_tokens:  int
    cost_usd:    float
    latency_ms:  float

class CostTracker:
    """
    Wrap the Anthropic client to record cost on every call.
    Drop-in replacement for client.messages.create().
    """
    def __init__(self):
        self.records = []

    def call(self, task_type: str, model: str, messages: list, **kwargs) -> object:
        """Make a tracked API call. Records model, tokens, cost, latency."""
        t0 = time.time()
        resp = client.messages.create(model=model, messages=messages, **kwargs)
        ms   = (time.time() - t0) * 1000

        usage       = resp.usage
        cache_read  = getattr(usage, "cache_read_input_tokens", 0) or 0
        cache_write = getattr(usage, "cache_creation_input_tokens", 0) or 0
        cost        = calc_cost(model, usage.input_tokens, usage.output_tokens,
                                cache_read, cache_write)

        self.records.append(CallRecord(
            timestamp  = datetime.now().strftime("%H:%M:%S"),
            model      = model,
            task_type  = task_type,
            in_tokens  = usage.input_tokens,
            out_tokens = usage.output_tokens,
            cost_usd   = cost,
            latency_ms = ms,
        ))
        return resp

    def report(self):
        """Print a cost breakdown by model and by task type."""
        if not self.records:
            print("No calls recorded yet.")
            return

        total_cost = sum(r.cost_usd for r in self.records)
        print(f"\n{'='*55}")
        print(f"  COST REPORT — {len(self.records)} calls, ${total_cost:.4f} total")
        print(f"{'='*55}")

        # By model
        by_model = {}
        for r in self.records:
            by_model.setdefault(r.model, {"calls": 0, "cost": 0.0})
            by_model[r.model]["calls"] += 1
            by_model[r.model]["cost"]  += r.cost_usd
        print("\nBy model:")
        for m, d in by_model.items():
            pct = d["cost"] / total_cost * 100
            print(f"  {m.split('-')[1]:<8} {d['calls']:>3} calls  ${d['cost']:.4f}  ({pct:.0f}%)")

        # By task type
        by_task = {}
        for r in self.records:
            by_task.setdefault(r.task_type, {"calls": 0, "cost": 0.0})
            by_task[r.task_type]["calls"] += 1
            by_task[r.task_type]["cost"]  += r.cost_usd
        print("\nBy task type (sorted by cost):")
        for task, d in sorted(by_task.items(), key=lambda x: -x[1]["cost"]):
            pct = d["cost"] / total_cost * 100
            print(f"  {task:<22} {d['calls']:>3} calls  ${d['cost']:.4f}  ({pct:.0f}%)")

# ── Run some tracked calls ────────────────────────────────────────────────────
tracker = CostTracker()
SAMPLE_CONFIG = """hostname branch-01
enable secret cisco123
line vty 0 4
 transport input all
snmp-server community public RO"""

print("Making 4 tracked calls (2 task types, 2 models)...\n")

for i in range(2):
    tracker.call("log_classify",    HAIKU,  max_tokens=20,  temperature=0,
                 messages=[{"role":"user","content": f"Is this a security alert? Reply YES or NO only. Log: 'BGP session down on Gi0/0'"}])
    tracker.call("security_audit",  SONNET, max_tokens=200, temperature=0,
                 messages=[{"role":"user","content": f"Audit this config for security issues. Return 1-2 key findings only.\n{SAMPLE_CONFIG}"}])

tracker.report()


### The Cost Formula You Need to Memorize

```
Cost per call = (input_tokens × input_rate + output_tokens × output_rate) / 1,000,000
```

Two things control your bill:
1. **Which model you pick** (Haiku is 5× cheaper than Sonnet per token)
2. **How many output tokens you allow** (output costs 3–5× more than input)

> **Network analogy:** This is the same math as bandwidth billing.
> Input tokens = ingress traffic. Output tokens = egress traffic.
> Egress always costs more — just like cloud providers charge more for outbound data.

---
## Demo 3 — Prompt Optimization: Cut Tokens Before They Hit the API

**The cheapest token is the one you never send.**

LLMs don't need politeness. They don't respond better to
*"Please kindly analyze the following and provide a thorough report..."*
than they do to *"Analyze this. Find issues. Return JSON."*

The savings per call are small. Multiplied across thousands of calls per month — significant.

Also: output tokens cost **5×** more than input tokens.
Asking for JSON instead of prose cuts output size by 60-70%.


In [ ]:
VERBOSE_PROMPT = """I would like you to please analyze this network configuration
file and tell me if there are any security vulnerabilities or issues that I should
be concerned about. Please be thorough in your analysis and make sure to check for
things like weak passwords, insecure protocols, missing security features, and
anything else that could be a problem.

Configuration:
{config}

Please provide a detailed report of your findings including severity levels and
recommendations for how to fix each issue you find. Thank you very much!"""

TIGHT_PROMPT = """Analyze this Cisco IOS config for security issues.
Check: weak auth, insecure protocols, missing hardening.
Return JSON: [{{"sev":"CRITICAL|HIGH|MEDIUM","issue":"...","fix":"..."}}]

Config:
{config}"""

CONFIG = SAMPLE_CONFIG   # reuse from Demo 2

# Count tokens for both prompts (no API call needed — just count_tokens)
def count(prompt_text: str) -> int:
    r = client.messages.count_tokens(
        model=HAIKU,
        messages=[{"role": "user", "content": prompt_text}]
    )
    return r.input_tokens

verbose_tok = count(VERBOSE_PROMPT.replace("{config}", CONFIG))
tight_tok   = count(TIGHT_PROMPT.replace("{config}", CONFIG))
saved_tok   = verbose_tok - tight_tok
monthly_calls = 12_000

print("=== Prompt Token Comparison ===\n")
print(f"  Verbose prompt:  {verbose_tok:>5} input tokens")
print(f"  Tight prompt:    {tight_tok:>5} input tokens")
print(f"  Tokens saved:    {saved_tok:>5} per call ({saved_tok/verbose_tok*100:.0f}% reduction)")
print()
monthly_saved = saved_tok * monthly_calls / 1_000_000 * PRICING[SONNET]["in"]
print(f"  At {monthly_calls:,} calls/month on Sonnet: ${monthly_saved:.2f}/month saved")
print(f"  For 20 prompts in your app:       ${monthly_saved*20:.2f}/month saved")
print()

# Now compare actual output length: verbose instruction → prose response vs JSON
print("=== Output Token Comparison ===\n")
resp_verbose = client.messages.create(
    model=HAIKU, max_tokens=300, temperature=0,
    messages=[{"role":"user","content": VERBOSE_PROMPT.replace("{config}", CONFIG)}]
)
resp_tight = client.messages.create(
    model=HAIKU, max_tokens=300, temperature=0,
    messages=[{"role":"user","content": TIGHT_PROMPT.replace("{config}", CONFIG)}]
)

v_out = resp_verbose.usage.output_tokens
t_out = resp_tight.usage.output_tokens
print(f"  Verbose instruction → prose response:  {v_out} output tokens")
print(f"  Tight + JSON instruction → JSON:       {t_out} output tokens")
print(f"  Output reduction: {v_out - t_out} tokens ({(v_out-t_out)/v_out*100:.0f}%)")

out_saved_usd = (v_out - t_out) / 1_000_000 * PRICING[HAIKU]["out"] * monthly_calls
print(f"  Output savings at {monthly_calls:,} calls/month: ${out_saved_usd:.2f}/month")
print()
print("Remember: output tokens cost 5× more than input tokens.")
print("JSON is shorter than prose AND costs less per token to generate.")


---
## Demo 4 — Model Routing: The LLM Cascade

**The highest-leverage optimization: route tasks to the cheapest model that handles them.**

Three tiers — same idea as QoS queues:
| Task | Model | Cost vs Sonnet | Examples |
|------|-------|---------------|----------|
| Simple | Haiku | 0.2x | Extract IPs, Yes/No classify, validate format |
| Medium | Sonnet | 1x | Security audit, BGP troubleshoot, comparison |
| Complex | Opus | 5x | Architecture design, strategic recommendations |

**The cascade approach**: start with Haiku. If it's confident → return.
If not → escalate to Sonnet. Only use Opus when Sonnet is uncertain.
Result: 60-70% of calls resolve at Haiku price.


In [ ]:
# ── Step 1: keyword heuristic (free — no API call) ──────────────────────────
SIMPLE_KEYWORDS  = ["classify","extract","validate","yes or no","list all",
                    "count","format","parse","is this","pull all","find all"]
COMPLEX_KEYWORDS = ["design","architect","root cause","compare and recommend",
                    "security strategy","optimize the entire","why is this"]

def classify_by_keywords(task: str) -> str:
    """Fast classification with no API call — handles ~70% of cases."""
    t = task.lower()
    if any(k in t for k in SIMPLE_KEYWORDS):
        return "simple"
    if any(k in t for k in COMPLEX_KEYWORDS):
        return "complex"
    return "unknown"   # needs AI classification

# ── Step 2: AI classifier (Haiku — cheap even if used for routing) ────────────
def classify_by_ai(task: str) -> str:
    """Use Haiku to classify complexity. max_tokens=5 → essentially free."""
    prompt = f"""Classify this task as one word: simple, medium, or complex.

simple  = extract/validate/yes-no/format/parse
medium  = analyze/troubleshoot/compare/summarize
complex = design/architect/root-cause/strategic

Task: "{task}"

Answer (one word only):"""

    resp = client.messages.create(
        model=HAIKU, max_tokens=5, temperature=0,
        messages=[{"role":"user","content": prompt}]
    )
    return resp.content[0].text.strip().lower().split()[0]

def get_model(task: str) -> tuple:
    """Classify task and return (model, complexity_label)."""
    label = classify_by_keywords(task)
    if label == "unknown":
        label = classify_by_ai(task)

    model_map = {"simple": HAIKU, "medium": SONNET, "complex": OPUS}
    return model_map.get(label, SONNET), label

# ── Step 3: Route and call ────────────────────────────────────────────────────
def route_call(task: str, content: str, max_tokens: int = 300) -> dict:
    """Route to the cheapest model that handles this task."""
    model, label = get_model(task)

    resp = client.messages.create(
        model=model, max_tokens=max_tokens, temperature=0,
        messages=[{"role":"user","content": f"{task}\n\n{content}"}]
    )

    actual_cost = calc_cost(model, resp.usage.input_tokens, resp.usage.output_tokens)
    sonnet_cost = calc_cost(SONNET, resp.usage.input_tokens, resp.usage.output_tokens)
    saved_pct   = (sonnet_cost - actual_cost) / sonnet_cost * 100 if sonnet_cost > 0 else 0

    return {
        "result":     resp.content[0].text.strip(),
        "model":      model.split("-")[1],
        "complexity": label,
        "cost_usd":   actual_cost,
        "saved_vs_sonnet_pct": saved_pct,
    }

# ── Test on four real networking tasks ────────────────────────────────────────
tasks = [
    ("Extract all IP addresses from this config. List only IPs, one per line.",
     "interface Gi0/0\n ip address 10.0.0.1 255.255.255.0\ninterface Gi0/1\n ip address 10.0.1.1 255.255.255.0"),
    ("Is this log line a BGP error? Reply YES or NO only.",
     "%BGP-5-ADJCHANGE: neighbor 10.1.1.1 Up"),
    ("Analyze this config for security vulnerabilities and explain each one.",
     SAMPLE_CONFIG),
    ("Design a network architecture for a multi-region active-active data center with OSPF and BGP.",
     "Requirements: 3 regions, sub-100ms failover, no single point of failure"),
]

print(f"{'Task (truncated)':<45} {'Model':<8} {'Complexity':<10} {'Cost':>8} {'Saved%':>7}")
print("─" * 83)
for task_desc, content in tasks:
    r = route_call(task_desc, content)
    print(f"{task_desc[:44]:<45} {r['model']:<8} {r['complexity']:<10} "
          f"${r['cost_usd']:>7.5f} {r['saved_vs_sonnet_pct']:>6.0f}%")

print()
print("Simple tasks automatically route to Haiku (0.2x cost of Sonnet).")
print("Complex tasks go to Sonnet or Opus — where the intelligence is needed.")


---
## Demo 5 — Batch API: 50% Discount for Non-Urgent Jobs

**Trade latency for cost. Perfect for nightly fleet audits.**

| | Real-Time API | Batch API |
|--|--|--|
| Response time | Seconds | 1–4 hours |
| Cost | Full price | **50% off** |
| Best for | Interactive use, troubleshooting | Scheduled jobs, bulk analysis |

Nightly audit of 400 routers: runs at 2 AM, results ready by 6 AM.
Who cares if it takes 4 hours? You save 50% every night, forever.

> Networking analogy: batch API = store-and-forward vs cut-through switching.
> Cut-through (real-time) = minimum latency, full cost.
> Store-and-forward (batch) = some latency, but you get to inspect and optimize.


In [ ]:
# ── Show the cost math ───────────────────────────────────────────────────────
print("=== Batch vs Real-Time Cost Comparison ===\n")
for num_devices in [10, 100, 400]:
    in_tok  = 2_500   # tokens per audit (config + prompt)
    out_tok =   400   # tokens per finding

    realtime = num_devices * calc_cost(HAIKU, in_tok, out_tok)
    batch    = realtime * 0.50   # 50% discount

    print(f"  {num_devices:>4} devices/night: real-time ${realtime:.2f}  batch ${batch:.2f}  "
          f"save ${realtime-batch:.2f}/night  (${(realtime-batch)*30:.2f}/month)")

print()
print("At 400 devices/night: $18/month saved just from switching to batch API.\n")

# ── How to use the Batch API ─────────────────────────────────────────────────
print("=== Batch API Code Pattern ===\n")
print("""
# 1. Build the batch request list
devices = [("router-01", config_01), ("router-02", config_02), ...]

requests = [
    {
        "custom_id": f"audit_{device_name}",   # your ID to match results later
        "params": {
            "model": HAIKU,
            "max_tokens": 600,
            "temperature": 0,
            "messages": [{"role": "user",
                          "content": f"Audit {device_name}. Return JSON findings.\n{config}"}]
        }
    }
    for device_name, config in devices
]

# 2. Submit — returns immediately with a batch_id (no waiting)
batch = client.messages.batches.create(requests=requests)
print(f"Batch submitted: {batch.id}")
print(f"Check back in 1-4 hours. Meanwhile, go home.")

# 3. Poll for completion (run this later, or in a scheduled job)
while True:
    status = client.messages.batches.retrieve(batch.id)
    print(f"Status: {status.processing_status} — "
          f"{status.request_counts.succeeded}/{status.request_counts.processing} done")
    if status.processing_status == "ended":
        break
    time.sleep(60)   # check every minute

# 4. Collect results
results = {}
for result in client.messages.batches.results(batch.id):
    device = result.custom_id.replace("audit_", "")
    if result.result.type == "succeeded":
        raw = result.result.message.content[0].text
        start = raw.find("{"); end = raw.rfind("}") + 1
        results[device] = json.loads(raw[start:end]) if start >= 0 else {}
""")

print("\nKey rule: Use batch for anything that doesn't need a response in under 1 minute.")
print("Nightly audits, weekly compliance, bulk documentation = batch. Interactive = real-time.")


---
## Demo 6 — Semantic Caching: Same Question Twice = Zero Cost

**10 engineers ask the same BGP question this week. Pay once. Serve 9 times free.**

A regular key-value cache only matches exact strings.
*"How do I configure BGP auth"* and *"BGP MD5 authentication setup"* are different strings.
A semantic cache recognizes them as the same question using vector similarity.

> Networking analogy: ARP cache.
> You don't broadcast an ARP request every time you need to reach 192.168.1.1.
> You cache the MAC → IP mapping. Same idea: cache question → answer.


In [ ]:
def cosine_sim(a: list, b: list) -> float:
    """Cosine similarity between two vectors. 1.0 = identical, 0.0 = unrelated."""
    dot   = sum(x*y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot / (mag_a * mag_b) if mag_a * mag_b > 0 else 0.0

class SemanticCache:
    """
    Cache LLM responses. On lookup, find the most similar stored question.
    If similarity >= threshold, return cached answer (cost = $0).
    If similarity < threshold, call the API and store the new answer.

    Threshold 0.92: same question worded differently = cache HIT.
    Threshold 0.70: loosely related questions = cache HIT (less conservative).
    """

    def __init__(self, threshold: float = 0.85):
        self.threshold = threshold
        self.store = []     # list of {embedding, question, answer, cost}
        self.hits   = 0
        self.misses = 0
        self.saved_usd = 0.0

    def _embed(self, text: str) -> list:
        """
        Production: use Voyage AI (voyage-3-lite) — Anthropic's recommended embedder.
        Here: character frequency vector (16-dim) for a zero-dependency demo.
        Not for production — just to illustrate the cache mechanism.
        """
        import hashlib
        h = hashlib.sha256(text.lower().strip().encode()).digest()
        return [b / 255.0 for b in h[:16]]

    def lookup(self, question: str):
        """Return cached answer if a similar question exists, else None."""
        if not self.store:
            return None
        qv   = self._embed(question)
        best = max(self.store, key=lambda e: cosine_sim(qv, e["embedding"]))
        sim  = cosine_sim(qv, best["embedding"])
        if sim >= self.threshold:
            self.hits      += 1
            self.saved_usd += best["cost"]
            return best["answer"]
        return None

    def store_answer(self, question: str, answer: str, cost: float = 0.003):
        self.misses += 1
        self.store.append({
            "embedding": self._embed(question),
            "question":  question,
            "answer":    answer,
            "cost":      cost,
        })

    def stats(self) -> str:
        total = self.hits + self.misses
        rate  = self.hits / total * 100 if total else 0
        return (f"Cache: {len(self.store)} entries | "
                f"{self.hits} hits / {total} lookups ({rate:.0f}% hit rate) | "
                f"${self.saved_usd:.4f} saved")

def cached_ask(question: str, cache: SemanticCache,
               model: str = HAIKU, max_tokens: int = 200) -> dict:
    """Ask with caching. Returns answer + whether it came from cache or API."""
    cached = cache.lookup(question)
    if cached:
        return {"answer": cached, "source": "CACHE", "cost": 0.0}

    resp = client.messages.create(
        model=model, max_tokens=max_tokens, temperature=0,
        messages=[{"role": "user", "content": question}]
    )
    answer = resp.content[0].text.strip()
    cost   = calc_cost(model, resp.usage.input_tokens, resp.usage.output_tokens)
    cache.store_answer(question, answer, cost)
    return {"answer": answer, "source": "API", "cost": cost}

# ── Simulate 10 engineers asking similar networking questions ─────────────────
cache = SemanticCache(threshold=0.85)

queries = [
    "How do I configure SSH on Cisco IOS?",                    # original
    "How do I configure SSH on Cisco IOS?",                    # exact duplicate
    "What is the command to enable SSH on a Cisco router?",    # similar wording
    "What is OSPF and how does it work?",                      # new topic
    "Explain OSPF routing protocol for beginners",             # similar to above
    "How do I set up BGP between two routers?",                # new topic
    "BGP configuration steps between two Cisco routers",       # similar to above
]

print(f"{'Query (truncated)':<48} {'Source':<7} {'Cost':>8}")
print("─" * 66)
for q in queries:
    r = cached_ask(q, cache, model=HAIKU)
    print(f"{q[:47]:<48} {r['source']:<7} ${r['cost']:>7.5f}")

print()
print(cache.stats())
print()
print("In a NOC with 20 engineers asking similar questions:")
print(f"  Estimated hit rate after warm-up: 40-60%")
print(f"  At $0.003/call and 500 calls/day: up to ${0.003 * 500 * 0.5 * 30:.0f}/month saved")


---
## Demo 7 — Budget Guard: Hard Daily Limits Per User

**Without limits, one runaway loop burns your monthly budget in hours.**

The $8,500 bill had a 12% contribution from a retry bug.
One process, stuck in a loop, hammering the API at $0.015/call.
A budget guard stops this before it becomes a problem.

> Networking analogy: CoPP (Control Plane Policing).
> You define a rate limit per traffic source.
> Traffic over the limit is dropped — not queued, not passed — dropped.
> Same here: API calls over the daily limit are blocked with an error.


In [ ]:
class BudgetGuard:
    """
    Per-user daily spending limit.
    Blocks API calls when limit is reached.
    Resets every 24 hours (or on restart for this demo).
    """

    def __init__(self):
        self.limits = {}   # user_id → daily_limit_usd
        self.spent  = {}   # user_id → spent_today_usd
        self.blocked = {}  # user_id → blocked_call_count

    def set_limit(self, user_id: str, daily_usd: float):
        """Register a user with a daily spending limit."""
        self.limits[user_id]  = daily_usd
        self.spent[user_id]   = 0.0
        self.blocked[user_id] = 0

    def check(self, user_id: str) -> tuple:
        """Return (allowed, remaining_usd, pct_used)."""
        limit     = self.limits.get(user_id, float("inf"))
        spent     = self.spent.get(user_id, 0.0)
        remaining = limit - spent
        pct       = spent / limit * 100 if limit > 0 else 100
        return remaining > 0, remaining, pct

    def record(self, user_id: str, cost: float):
        """Record a successful call's cost."""
        self.spent[user_id] = self.spent.get(user_id, 0.0) + cost

    def block(self, user_id: str):
        """Record a blocked call."""
        self.blocked[user_id] = self.blocked.get(user_id, 0) + 1

    def guarded_call(self, user_id: str, model: str,
                     messages: list, **kwargs) -> dict:
        """
        Make an API call — but only if the user has budget remaining.
        Returns {ok, result, cost, message}.
        """
        allowed, remaining, pct = self.check(user_id)

        if not allowed:
            self.block(user_id)
            return {
                "ok":      False,
                "result":  None,
                "cost":    0.0,
                "message": f"BLOCKED: {user_id} daily limit reached "
                           f"(${self.spent[user_id]:.4f} / ${self.limits[user_id]:.2f})"
            }

        # Warn when approaching limit
        if pct > 80:
            print(f"  ⚠️  [{user_id}] budget at {pct:.0f}% — ${remaining:.4f} remaining")

        resp = client.messages.create(model=model, messages=messages, **kwargs)
        cost = calc_cost(model, resp.usage.input_tokens, resp.usage.output_tokens)
        self.record(user_id, cost)

        return {
            "ok":      True,
            "result":  resp.content[0].text.strip(),
            "cost":    cost,
            "message": f"OK — spent ${cost:.5f}, total ${self.spent[user_id]:.5f}"
        }

    def status(self, user_id: str) -> str:
        limit   = self.limits.get(user_id, 0)
        spent   = self.spent.get(user_id, 0)
        blocked = self.blocked.get(user_id, 0)
        pct     = spent / limit * 100 if limit > 0 else 0
        bar     = "█" * int(pct / 10) + "░" * (10 - int(pct / 10))
        return (f"{user_id:<12} [{bar}] {pct:>5.1f}%  "
                f"${spent:.5f} / ${limit:.3f}  blocked={blocked}")

# ── Simulate two users with different limits ──────────────────────────────────
guard = BudgetGuard()
guard.set_limit("dev-alice", 0.001)   # very tight — triggers guard quickly in demo
guard.set_limit("dev-bob",   0.010)   # relaxed

questions = [
    ("dev-alice", "Is 'telnet' an insecure protocol? Reply YES or NO."),
    ("dev-alice", "Should VTY lines use SSH? Reply YES or NO."),
    ("dev-alice", "Is SNMP v2c secure? Reply YES or NO."),   # may be blocked
    ("dev-bob",   "List 3 Cisco hardening best practices. Be concise."),
    ("dev-bob",   "What does 'service password-encryption' do?"),
]

print("Simulating guarded API calls:\n")
for user, question in questions:
    r = guard.guarded_call(user, HAIKU,
                           messages=[{"role":"user","content": question}],
                           max_tokens=30, temperature=0)
    status = "✅" if r["ok"] else "🚫"
    answer = (r["result"] or "BLOCKED")[:60] if r["ok"] else "BLOCKED"
    print(f"  {status} [{user}] {question[:45]}...")
    print(f"       → {answer} | {r['message']}\n")

print("=== Budget Status ===")
for user in ["dev-alice", "dev-bob"]:
    print(f"  {guard.status(user)}")


---
## Summary

**The five strategies that took a $8,500 bill to $2,100:**

| Strategy | Typical Savings | Effort |
|----------|----------------|--------|
| 1. Prompt optimization | 5-20% | Low — one-time rewrite |
| 2. Model routing (cascade) | 40-70% | Medium — add classifier |
| 3. Batch API | 50% on eligible jobs | Low — change API call |
| 4. Semantic caching | 30-60% in NOC/chatbot | Medium — add cache layer |
| 5. Budget guards | Prevents runaway spend | Low — wrap every call |

**The QoS mindset applied to AI costs:**

```
Network QoS                    AI Cost Optimization
─────────────────────────────────────────────────────────────────
DSCP marking at edge      ──→  Classify task complexity
Priority queue (LLQ)      ──→  Complex tasks → Opus/Sonnet
Best-effort queue         ──→  Simple tasks → Haiku
Rate limiting (policing)  ──→  Budget guard per user/team
NetFlow monitoring        ──→  CostTracker per feature
ARP cache                 ──→  Semantic cache for repeat queries
```

> The lesson from the $8,500 bill:
> 89% of that spend was waste. The fix took two weeks.
> Visibility first. Then routing. Then caching. Then limits.
> In that order.
